# 4 - Raw harm vs. vehicle-exposure-adjusted harm

The goal is to see whether the locations that look worst by raw injury counts are the same locations that look worst after accounting for motor-vehicle traffic volume.

In [ ]:
from pathlib import Path
import sys

# Make imports work whether Jupyter starts in the repo root or notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

CACHE_DIR = ROOT / "data" / "cache"
OUTPUT_DIR = ROOT / "data" / "outputs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_START = "2021-01-01"
ANALYSIS_END = "2025-12-31"
REFRESH = False

print("Repo root:", ROOT)
print("Analysis period:", ANALYSIS_START, "to", ANALYSIS_END)
print("Force API refresh:", REFRESH)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.api import load_cache
from src.joins import add_rank_comparison


In [ ]:
segments = load_cache("segments_with_exposure", cache_dir=CACHE_DIR)
if segments is None:
    raise RuntimeError("Run notebooks 01-03 first, or execute notebook 03 to create segments_with_exposure.")

ranked = add_rank_comparison(
    segments,
    raw_metric="people_injured",
    adjusted_metric="injuries_per_100m_vmt",
)
print(f"Segments: {len(ranked):,}")
print(f"Rate eligible: {ranked['rate_eligible'].sum():,}")


## Top segments by raw injuries

In [ ]:
show_cols = [c for c in [
    "segmentid", "street", "people_injured", "people_killed", "aadt",
    "injuries_per_100m_vmt", "raw_rank", "adjusted_rank", "rank_change",
    "aadt_match_confidence"
] if c in ranked.columns]

display(ranked.sort_values("people_injured", ascending=False)[show_cols].head(25))


## Top segments by injuries per 100M VMT

For exploration we require a high/medium AADT correspondence. **Do not use this raw rate alone !!!!!!!:** short or low-exposure segments can have unstable rates. **TODO** add minimum-data rules and statistical shrinkage / Empirical Bayes.

In [ ]:
eligible = ranked[ranked["rate_eligible"]].copy()
display(eligible.sort_values("injuries_per_100m_vmt", ascending=False)[show_cols].head(25))


## Biggest upward movers after traffic adjustment

A positive `rank_change` means a segment becomes **more concerning** after adjusting for vehicle exposure.

In [ ]:
display(
    eligible.sort_values("rank_change", ascending=False)[show_cols].head(25)
)


## Exploratory maps

These maps answer different questions: total harm vs. harm relative to vehicle exposure.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ranked.plot(column="people_injured", linewidth=0.8, ax=ax, legend=True)
ax.set_title(f"Raw injuries by LION segment, {ANALYSIS_START[:4]}–{ANALYSIS_END[:4]}")
ax.set_axis_off()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
eligible.plot(column="injuries_per_100m_vmt", linewidth=0.8, ax=ax, legend=True)
ax.set_title("Injuries per 100M vehicle-miles traveled (high/medium AADT matches)")
ax.set_axis_off()
plt.show()


## Export a compact review table

The CSV intentionally omits geometry so it is easy to inspect in Excel/Google Sheets. The full GeoDataFrame remains in the cached snapshot.

In [ ]:
export_cols = [c for c in [
    "segmentid", "street", "crashes", "people_injured", "people_killed",
    "pedestrians_injured", "cyclists_injured", "aadt", "roadway_name",
    "aadt_distance_ft", "aadt_name_match", "aadt_match_confidence",
    "length_miles", "vehicle_miles_traveled", "injuries_per_100m_vmt",
    "raw_rank", "adjusted_rank", "rank_change"
] if c in ranked.columns]

out_path = OUTPUT_DIR / "raw_vs_exposure.csv"
ranked[export_cols].sort_values("adjusted_rank").to_csv(out_path, index=False)
print("Wrote", out_path)
